# Cortex Code — Usage & Cost Analysis (CLI + Snowsight)

Queries `SNOWFLAKE.ACCOUNT_USAGE` to surface usage patterns, credit consumption, and projected costs for both Cortex Code surfaces:
- **CLI** — `CORTEX_CODE_CLI_USAGE_HISTORY` (terminal / `snow` command)
- **Snowsight** — `CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY` (browser IDE)

**AI Credit price:** $2.00 per credit (global on-demand, April 2026 consumption table).  
Capacity customers: adjust `ai_credit_price` in the config cell below.

**Required privilege:**
```sql
GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE <your_role>;
```

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

ai_credit_price = 2.00

source = 'both'  # Options: 'cli', 'snowsight', 'both'

CLI_TABLE       = 'SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY'
SNOWSIGHT_TABLE = 'SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY'

if source == 'cli':
    source_table = CLI_TABLE
elif source == 'snowsight':
    source_table = SNOWSIGHT_TABLE
else:
    source_table = f"""(
        SELECT 'cli'       AS source, * FROM {CLI_TABLE}
        UNION ALL
        SELECT 'snowsight' AS source, * FROM {SNOWSIGHT_TABLE}
    ) t"""

print(f"AI Credit price : ${ai_credit_price:.2f} per credit")
print(f"Source          : {source}")
print("Adjust ai_credit_price or source above before running all cells.")

## 1. Daily Usage Summary
Request counts, tokens, credits, and estimated dollar cost per day — last 30 days.

In [ ]:
daily_usage = session.sql(f"""
    SELECT
        USAGE_TIME::DATE                        AS usage_date,
        COUNT(*)                                AS total_requests,
        SUM(TOKENS)                             AS total_tokens,
        ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
        ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd
    FROM {source_table}
    GROUP BY 1
    ORDER BY 1 DESC
    LIMIT 30
""").to_pandas()
daily_usage

## 2. Weekly Usage Trend
Aggregates by week — useful for spotting adoption ramp and seasonal patterns.

In [ ]:
weekly_trend = session.sql(f"""
    SELECT
        DATE_TRUNC('WEEK', USAGE_TIME)          AS week_start,
        COUNT(*)                                AS total_requests,
        SUM(TOKENS)                             AS total_tokens,
        ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
        ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd,
        ROUND(AVG(TOKENS), 0)                   AS avg_tokens_per_request
    FROM {source_table}
    GROUP BY 1
    ORDER BY 1 DESC
    LIMIT 13
""").to_pandas()
weekly_trend

## 3. Top Users by Credits
Ranks all users by total AI credit consumption, with first/last usage timestamps.

In [ ]:
top_users = session.sql(f"""
    SELECT
        USER_ID,
        COUNT(*)                             AS total_requests,
        SUM(TOKENS)                             AS total_tokens,
        ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
        ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd,
        MIN(USAGE_TIME)                         AS first_usage,
        MAX(USAGE_TIME)                         AS last_usage
    FROM {source_table}
    GROUP BY 1
    ORDER BY total_credits DESC
    LIMIT 20
""").to_pandas()
top_users

## 4. Hourly Usage Pattern
Reveals which hours of the day see peak activity — all-time aggregate.

In [ ]:
hourly_pattern = session.sql(f"""
    SELECT
        HOUR(USAGE_TIME)                        AS hour_of_day,
        COUNT(*)                                AS total_requests,
        SUM(TOKENS)                             AS total_tokens,
        ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits
    FROM {source_table}
    GROUP BY 1
    ORDER BY 1
""").to_pandas()
hourly_pattern

## 5. Usage by Model
Breaks out credits by model using the `CREDITS_GRANULAR` variant column, with cache read/write split.

In [ ]:
usage_by_model = session.sql(f"""
    SELECT
        f.key                                                   AS model,
        COUNT(*)                                                AS total_requests,
        SUM(NVL(f.value:cache_read_input::FLOAT, 0))            AS cache_read_credits,
        SUM(NVL(f.value:cache_write_input::FLOAT, 0))           AS cache_write_credits,
        SUM(NVL(f.value:input::FLOAT, 0))                       AS input_credits,
        SUM(NVL(f.value:output::FLOAT, 0))                      AS output_credits,
        ROUND(
            SUM(NVL(f.value:cache_read_input::FLOAT, 0)
                + NVL(f.value:cache_write_input::FLOAT, 0)
                + NVL(f.value:input::FLOAT, 0)
                + NVL(f.value:output::FLOAT, 0)), 4)            AS total_credits,
        ROUND(
            SUM(NVL(f.value:cache_read_input::FLOAT, 0)
                + NVL(f.value:cache_write_input::FLOAT, 0)
                + NVL(f.value:input::FLOAT, 0)
                + NVL(f.value:output::FLOAT, 0)) * {ai_credit_price}, 2) AS estimated_cost_usd
    FROM {source_table},
        LATERAL FLATTEN(input => CREDITS_GRANULAR) f
    GROUP BY 1
    ORDER BY total_credits DESC
""").to_pandas()
usage_by_model

## 6. Cost Projections
Takes the 22 busiest working days on record (proxy for a full working month), then projects to weekly, monthly, and annual spend.

In [ ]:
cost_projections = session.sql(f"""
    WITH daily_costs AS (
        SELECT
            USAGE_TIME::DATE            AS usage_date,
            SUM(TOKEN_CREDITS)          AS daily_credits
        FROM {source_table}
        GROUP BY 1
        ORDER BY daily_credits DESC
        LIMIT 22
    ),
    stats AS (
        SELECT
            MIN(daily_credits)  AS min_daily,
            AVG(daily_credits)  AS mean_daily,
            MAX(daily_credits)  AS max_daily
        FROM daily_costs
    )
    SELECT 'Per Day'   AS period,
        ROUND(min_daily, 2)             AS min_credits,
        ROUND(mean_daily, 2)            AS mean_credits,
        ROUND(max_daily, 2)             AS max_credits,
        ROUND(min_daily * {ai_credit_price}, 2)      AS min_cost_usd,
        ROUND(mean_daily * {ai_credit_price}, 2)     AS mean_cost_usd,
        ROUND(max_daily * {ai_credit_price}, 2)      AS max_cost_usd
    FROM stats
    UNION ALL
    SELECT 'Per Week',
        ROUND(min_daily * 5, 2),  ROUND(mean_daily * 5, 2),  ROUND(max_daily * 5, 2),
        ROUND(min_daily * 5 * {ai_credit_price}, 2), ROUND(mean_daily * 5 * {ai_credit_price}, 2), ROUND(max_daily * 5 * {ai_credit_price}, 2)
    FROM stats
    UNION ALL
    SELECT 'Per Month',
        ROUND(min_daily * 22, 2), ROUND(mean_daily * 22, 2), ROUND(max_daily * 22, 2),
        ROUND(min_daily * 22 * {ai_credit_price}, 2), ROUND(mean_daily * 22 * {ai_credit_price}, 2), ROUND(max_daily * 22 * {ai_credit_price}, 2)
    FROM stats
    UNION ALL
    SELECT 'Per Year',
        ROUND(min_daily * 260, 2), ROUND(mean_daily * 260, 2), ROUND(max_daily * 260, 2),
        ROUND(min_daily * 260 * {ai_credit_price}, 2), ROUND(mean_daily * 260 * {ai_credit_price}, 2), ROUND(max_daily * 260 * {ai_credit_price}, 2)
    FROM stats
""").to_pandas()
cost_projections

## 7. CLI vs Snowsight Comparison
Side-by-side totals for each surface — always queries both sources regardless of the `source` setting above.

In [ ]:
source_comparison = session.sql(f"""
    WITH cli AS (
        SELECT
            'cli'                               AS surface,
            COUNT(*)                            AS total_requests,
            SUM(TOKENS)                         AS total_tokens,
            ROUND(SUM(TOKEN_CREDITS), 4)        AS total_credits,
            ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd
        FROM {CLI_TABLE}
    ),
    snowsight AS (
        SELECT
            'snowsight'                         AS surface,
            COUNT(*)                            AS total_requests,
            SUM(TOKENS)                         AS total_tokens,
            ROUND(SUM(TOKEN_CREDITS), 4)        AS total_credits,
            ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd
        FROM {SNOWSIGHT_TABLE}
    )
    SELECT * FROM cli
    UNION ALL
    SELECT * FROM snowsight
""").to_pandas()
source_comparison

## 8. Cortex Code Model Pricing Reference
Official rates from the Snowflake Service Consumption Table, **Table 6(e) — Cortex Code** (effective April 1, 2026).  
All rates are AI Credits per one million tokens. Multiply by your AI credit price to get USD cost per 1M tokens.

In [ ]:
models = [
    {"model": "claude-4-sonnet",   "input": 1.50, "output": 7.50,  "cache_write": 1.88, "cache_read": 0.15},
    {"model": "claude-opus-4-5",   "input": 2.75, "output": 13.75, "cache_write": 3.44, "cache_read": 0.28},
    {"model": "claude-opus-4-6",   "input": 2.75, "output": 13.75, "cache_write": 3.44, "cache_read": 0.28},
    {"model": "claude-sonnet-4-5", "input": 1.65, "output": 8.25,  "cache_write": 2.07, "cache_read": 0.17},
    {"model": "claude-sonnet-4-6", "input": 1.65, "output": 8.25,  "cache_write": 2.07, "cache_read": 0.17},
    {"model": "openai-gpt-5.2",    "input": 0.97, "output": 7.70,  "cache_write": None, "cache_read": 0.10},
    {"model": "openai-gpt-5.4",    "input": 1.38, "output": 8.25,  "cache_write": None, "cache_read": 0.14},
]

df = pd.DataFrame(models)
df.columns = [
    "Model",
    "Input (AI Credits/1M tokens)",
    "Output (AI Credits/1M tokens)",
    "Cache Write (AI Credits/1M tokens)",
    "Cache Read (AI Credits/1M tokens)",
]

print(f"Source: Snowflake Service Consumption Table, Table 6(e) — Cortex Code (effective April 1, 2026)")
print(f"AI Credit On-Demand Price: ${ai_credit_price:.2f}/credit (global)")
print()
df

## 9. Governance & Spend Controls

Key Snowflake-native levers for controlling Cortex Code spend:

| Lever | Mechanism | Enforcement |
|-------|-----------|-------------|
| Account budget | `SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET` | Alert-only |
| Custom budget | `SNOWFLAKE.CORE.BUDGET` | Alert + stored proc actions |
| Budget actions | Threshold-triggered stored procedures | Configurable |
| Notifications | Email / SNS / Webhook | Push alert |
| Model selection | Steer to lower-cost models via `config.toml` | Policy / guidance |
| RBAC | `SNOWFLAKE.CORTEX_USER` database role | Hard enforcement |

See the **Governance** tab in the Streamlit app (`streamlit_app.py`) for copy-paste SQL for each lever.

### Quick Reference: Account Budget
```sql
USE ROLE ACCOUNTADMIN;
CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!ACTIVATE();
CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!SET_SPENDING_LIMIT(1000);
CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!SET_EMAIL_NOTIFICATIONS('finops@company.com');
```

### Quick Reference: RBAC
```sql
-- Grant / revoke Cortex Code access
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE <developer_role>;
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE <developer_role>;
```

In [ ]:
print("=== Current Custom Budgets ===")
try:
    budgets = session.sql("SHOW SNOWFLAKE.CORE.BUDGET INSTANCES IN ACCOUNT").to_pandas()
    if budgets.empty:
        print("No custom budgets configured.")
    else:
        print(budgets.to_string(index=False))
except Exception as e:
    print(f"Could not query budgets (ACCOUNTADMIN or BUDGET_VIEWER required): {e}")